# OSNet: целевые проверки после аудита

Новый изолированный эксперимент. MVP, старые результаты и разметка организаторов не меняются.
По умолчанию запускается только **pilot**: один seed, пять условий, до 8 500 updates.
Это B0, прежняя и исправленная color-ветви, разрешение 256 и triplet. NiVe не используется.

**Restart Kernel → Run All** в прежнем `.venv`. Прогресс: условие, seed, шаг/бюджет, время и ETA.
Не запускай одновременно другой training notebook. Подробности и ограничения — в `README.md`.


In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/osnet_review_protocol.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из проекта Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
from IPython.display import Markdown, display
from training.osnet_ablation_suite import Budget
from training.osnet_review_protocol import prepare, run

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
print('Repository:', REPO, '| Torch:', torch.__version__)


## 1. Заранее фиксируем правила

`pilot` — только первый seed, без выбора победителя и outer-оценки.
После разбора пилота переключи только `PHASE = 'confirm_inner'`: ещё два primary seed,
выбор одного общего шага на рецепт, затем alternate на трёх seed строго на этом шаге.
Alternate не меняет победителя или длительность. Draws — не независимые training seed.

`final` — отдельный запуск после разбора confirmation; требует `ALLOW_OUTER_EVALUATION = True`.
Все final веса фиксируются до первой outer-оценки. Outer ранее использовалась для выбора MVP:
она остаётся development validation, а не новым независимым тестом.


In [ ]:
RUN_NAME = 'review_v1'
PHASE = 'final'  # 'pilot' -> 'confirm_inner' -> 'final', вручную после анализа
ALLOW_OUTER_EVALUATION = True
ALLOW_CPU_TRAINING = False
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
BUDGET = Budget(max_steps=1700, evaluation_interval=200, warmup_steps=100)
SEEDS = (20260915, 20260916, 20260917)
DRAW_SEEDS = (20260915, 20261016, 20261117)

print('Run:', RUN_NAME, '| phase:', PHASE, '| device:', DEVICE)
print('Training seeds:', SEEDS, '| fixed draws:', DRAW_SEEDS)
print('Pilot: 5 stages /', 5 * BUDGET.max_steps, 'updates')
print('Inner confirmation including pilot: at most 30 stages /', 30 * BUDGET.max_steps, 'updates')
print('No NiVe; no automatic next phase; no MVP replacement.')


## 2. Проверяем входы — обучение ещё не начинается

Хеши изображений, CSV, splits, исходников, stock/MVP и масок записываются в новый manifest.
При несовпадении существующего manifest не удаляй его и не обходи проверку.
Изменение кода, seed, budgets или runtime требует нового RUN_NAME; переключать PHASE можно.


In [ ]:
context = prepare(RUN_NAME, device=DEVICE, budget=BUDGET, seeds=SEEDS, draw_seeds=DRAW_SEEDS)
print('Output:', context['output'])
print('Fingerprint:', context['signature'])
for fold, partition in context['manifest']['inner'].items():
    print(fold, '| train IDs:', len(partition['train']), '| validation IDs:', len(partition['validation']))
print('Preflight passed. No optimizer updates yet.')


## 3. Запуск выбранного этапа

После прерывания снова **Restart Kernel → Run All** с теми же настройками.
Завершённые стадии проверяются и не переобучаются. Незавершённый блок до 200 updates повторится.
Каждый selectable checkpoint сохраняется: понадобится несколько гигабайт свободного места.

LR всегда рассчитан на 1700 updates, даже когда alternate/final останавливается раньше.
Это перенос правила числа updates, не воспроизведение старых 5 эпох MVP.


In [ ]:
if DEVICE == 'cpu' and not ALLOW_CPU_TRAINING:
    raise RuntimeError('CPU-обучение отключено. Выбери MPS/CUDA или явно разреши CPU.')
result = run(context, phase=PHASE, allow_outer=ALLOW_OUTER_EVALUATION)
print('Complete:', result['complete'], '| Outer evaluated:', result['outer_evaluated'], '| Promoted:', result['promoted'])


## 4. Результаты для разбора

Передай отчёт текущей фазы и одноимённый JSON. В confirmation смотри paired-seed разницы,
а не только максимальный mAP; исправленная ветвь сравнивается и с B0, и со старой color-ветвью.
На alternate сохраняются три draws, same-camera junk, confidence и автоматические masked diagnostics.
Эти маски **не подтверждены как plate-only**; по ним нельзя делать сильный вывод о зависимости от номеров.

Final создаёт experimental ONNX и frozen bundle. Экспорт submissions не требует train/calibration;
команды CPU/GPU и ограничения проверки скорости приведены в README. Ничего не копируется в MVP.


In [ ]:
report = context['output'] / f'{PHASE.upper()}_RESULTS.md'
display(Markdown(report.read_text(encoding='utf-8')))
print('Detailed results:', context['output'] / f'{PHASE}.json')
